# Tests des fonctions de calcul de la saturation

In [1]:
import datetime

import numpy as np
import pandas as pd
from saturation_image_quali import (
    to_sampled_sessions,
    to_sampled_state_grp,
    to_sampled_state_poc,
    to_sampled_statuses,
)

from saturation import (
    hysteresis,
    # to_sampled_sessions,
    # to_sampled_state_grp,
    # to_sampled_state_pdc,
    # to_sampled_statuses,
)


## Test échantillonage des sessions

In [2]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [2.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 2.1], [5.5, 7.5], [13.1, 15.1] -> [1.1, 2, 2]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6] -> [1.5, 2, 3.1, 2.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[0]['occupation_pdc'] == 'occupe'
assert sessions.iloc[1]['occupation_pdc'] == 'occupe'
assert sessions.iloc[5]['occupation_pdc'] == 'f_libre'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'
assert sessions.iloc[34]['occupation_pdc'] == 'occupe'


In [3]:
sessions = to_sampled_sessions(test, init, timestamp, echantillons, min_duration=datetime.timedelta(hours=1.2) )

assert sessions.iloc[1]['occupation_pdc'] == 'f_libre'

# sessions

In [4]:
sessions = to_sampled_sessions(test, init, timestamp, echantillons, max_duration=datetime.timedelta(hours=3) )

assert sessions.iloc[34]['occupation_pdc'] == 'f_libre'

# sessions

In [5]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [6.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']}) 
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 6.1], [5.5, 7.5], [13.1, 15.1]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[5]['occupation_pdc'] == 'occupe'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'

# sessions

## Test échantillonage des statuts

In [6]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
valeurs = [1, 1.2, 3, 3.5, 5, 6.1, 12] # p1 : [1, 3.5, 6.1] p2 : [1.2, 3, 5, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in valeurs],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'id_pdc_itinerance': pdc}) 
statuses = to_sampled_statuses(test, init, timestamp, echantillons)
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses = to_sampled_statuses(test, init, timestamp, echantillons, datetime.timedelta(hours=1.9))
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'en_service'

# statuses

## Test assemblage des sessions et des statuts

In [7]:
sessions = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'], 
                       'periode': [0,1,2,0,1,2,0,1,2],
                       'occupation_pdc': ['occupe', 'f_libre', 'occupe', 'f_libre', 'occupe', 'f_libre','f_libre', 'occupe', 'f_libre']})
status = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p3', 'p3', 'p3', 'p4', 'p4', 'p4'], 
                       'periode': [0,1,2,0,1,2, 0,1,2],
                       'etat_pdc': ['hors_service', 'hors_service', 'en_service', 'en_service', 'hors_service', 'hors_service', 'en_service', 'hors_service', 'en_service']})
merged = pd.merge(sessions, status, how='outer', on=['id_pdc_itinerance', 'periode']).fillna('aaa')
merged

,id_pdc_itinerance,periode,occupation_pdc,etat_pdc
0,p1,0,occupe,hors_service
1,p1,1,f_libre,hors_service
2,p1,2,occupe,en_service
3,p2,0,f_libre,aaa
4,p2,1,occupe,aaa
5,p2,2,f_libre,aaa
6,p3,0,f_libre,en_service
7,p3,1,occupe,hors_service
8,p3,2,f_libre,hors_service
9,p4,0,aaa,en_service


In [8]:
merged = to_sampled_state_poc(sessions, status)
assert list(merged['state'][0:4]) == ['occupe', 'hors_service', 'occupe', 'libre']
merged

,id_pdc_itinerance,periode,state
0,p1,0,occupe
1,p1,1,hors_service
2,p1,2,occupe
3,p2,0,libre
4,p2,1,occupe
5,p2,2,libre
6,p3,0,libre
7,p3,1,occupe
8,p3,2,hors_service
9,p4,0,libre


In [9]:
print(merged['state'])
merged['state'].str.replace('en_service', 'libre')
merged['state'] = merged['state'].str.replace('en_service', 'libre')
merged

0           occupe
1     hors_service
2           occupe
3            libre
4           occupe
5            libre
6            libre
7           occupe
8     hors_service
9            libre
10    hors_service
11           libre
Name: state, dtype: object


,id_pdc_itinerance,periode,state
0,p1,0,occupe
1,p1,1,hors_service
2,p1,2,occupe
3,p2,0,libre
4,p2,1,occupe
5,p2,2,libre
6,p3,0,libre
7,p3,1,occupe
8,p3,2,hors_service
9,p4,0,libre


## Test état global échantillonné d'un groupement de pdc

In [10]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'],
                     'periode' : [0, 1, 2, 0, 1, 2, 0, 1, 2],
                     'state' : ['occupe', 'hors_service', 'occupe', 'libre', 'occupe', 'libre', 'libre', 'occupe', 'hors_service']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3'],
                         'id_station_itinerance': ['s1', 's1', 's2']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

,id_station_itinerance,periode,occupe,hors_service,libre,nb_pdc,hs,inactif,sature,surcharge,actif,state
0,s1,0,1,0,1,2,False,False,False,False,True,3
1,s1,1,1,1,0,2,False,False,True,False,False,5
2,s1,2,1,0,1,2,False,False,False,False,True,3
3,s2,0,0,0,1,1,False,True,False,False,False,2
4,s2,1,1,0,0,1,False,False,True,False,False,5
5,s2,2,0,1,0,1,True,False,False,False,False,1


## Test de la pleine utilisation

In [11]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2',
                                           'p3', 'p3', 'p3',
                                           'p4', 'p4', 'p4',
                                           'p5', 'p5', 'p5',
                                           'p6', 'p6', 'p6'],
                     'periode' : [0, 1, 2] * 6,
                     'state' : ['libre', 'occupe', 'occupe'] * 6})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3', 'p4', 'p5', 'p6'],
                         'id_station_itinerance': ['s1'] *6}) 
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2)
res
assert res['sature'][2]
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True)
assert (res['sature'] == res['pleine_utilisation']).all()
test.loc[2,'state'] = 'libre'
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True)
assert not res['sature'][2] and res['pleine_utilisation'][2]
test.loc[5,'state'] = 'libre'
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True)
assert not res['sature'][2] and not res['pleine_utilisation'][2]
res

,id_station_itinerance,periode,occupe,hors_service,libre,nb_pdc,hs,inactif,sature,surcharge,actif,state,pleine_utilisation
0,s1,0,0,0,6,6,False,True,False,False,False,2,False
1,s1,1,6,0,0,6,False,False,True,False,False,5,True
2,s1,2,4,0,2,6,False,False,False,False,True,3,False


In [12]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2', 'p2', 'p2'],
                     'periode' : [0, 1, 2, 3, 4, 5,
                                  0, 1, 2, 3, 4, 5],
                     'state' : ['occupe', 'hors_service', 'occupe', 'occupe', 'hors_service', 'libre',
                                'libre', 'libre', 'occupe', 'hors_service', 'hors_service', 'libre']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2'],
                         'id_station_itinerance': ['s1', 's1']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

,id_station_itinerance,periode,occupe,hors_service,libre,nb_pdc,hs,inactif,sature,surcharge,actif,state
0,s1,0,1,0,1,2,False,False,False,False,True,3
1,s1,1,0,1,1,2,False,True,False,False,False,2
2,s1,2,2,0,0,2,False,False,True,False,False,5
3,s1,3,1,1,0,2,False,False,True,False,False,5
4,s1,4,0,2,0,2,True,False,False,False,False,1
5,s1,5,0,0,2,2,False,True,False,False,False,2


In [13]:
test = pd.DataFrame({'name':         ['hs', 'inactif', 'sature', 'surcharge', 'actif'],
                     'occupe':       [0, 0, 5, 3, 2],
                     'hors_service': [6, 2, 1, 2, 2],
                     'libre':        [0, 4, 0, 1, 2],
                     'nb_pdc':       [6, 6, 6, 6, 6]})
# 2e partie de la fonction : to_sampled_state_grp
test['hs'] = (test['libre'] + test['occupe'] == 0) & (test['hors_service'] > 0)
test['inactif'] = ~test['hs'] & (test['occupe'] == 0)
test['sature'] = ~test['hs'] & ~test['inactif'] & (test['libre']/test['nb_pdc'] < 0.1)
test['surcharge'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & (test['libre']/test['nb_pdc'] < 0.2)
test['actif'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & ~test['surcharge']
test['state'] = test['hs'] + test['inactif'] * 2 + test['actif'] * 3 + test['surcharge'] * 4 + test['sature'] * 5
test

,name,occupe,hors_service,libre,nb_pdc,hs,inactif,sature,surcharge,actif,state
0,hs,0,6,0,6,True,False,False,False,False,1
1,inactif,0,2,4,6,False,True,False,False,False,2
2,sature,5,1,0,6,False,False,True,False,False,5
3,surcharge,3,2,1,6,False,False,False,True,False,4
4,actif,2,2,2,6,False,False,False,False,True,3


## test de l'hysteresis

In [14]:
# seuil à 6 et 9
serie = pd.Series([1, 2, 5, 7, 5, 8, 10, 12, 8, 11, 8, 5, 7, 2])
res = hysteresis(serie, 6, 9)
assert res[6:10].all() == True

In [15]:
test = pd.DataFrame({'id_pdc_itinerance': ['pdc1'] * 10 + ['pdc2'] * 10,
                     'periode':      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] * 2,
                     'occupe':       [0, 1, 1, 0, 0, 1, 0, 0, 0, 1] * 2,
                     'hors_service': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0] * 2, 
                     'libre':        [1, 0, 0, 1, 1, 0, 1, 1, 1, 0] * 2,
                     'nb_pdc':       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] * 2})
hyst = 3
pleine_occupation = test['occupe'].copy()
for i in range(1, hyst):
    pleine_occupation += [0] * i + list(test['occupe'])[0:len(test) - i]
f_id_pdc_itinerance = pd.Series(['aucun'] * hyst + list(test['id_pdc_itinerance'])[0:len(test) - hyst])
test['valid_po'] = f_id_pdc_itinerance == test['id_pdc_itinerance']
test['po'] = pleine_occupation > 0
test

,id_pdc_itinerance,periode,occupe,hors_service,libre,nb_pdc,valid_po,po
0,pdc1,0,0,0,1,1,False,False
1,pdc1,1,1,0,0,1,False,True
2,pdc1,2,1,0,0,1,False,True
3,pdc1,3,0,0,1,1,True,True
4,pdc1,4,0,0,1,1,True,True
5,pdc1,5,1,0,0,1,True,True
6,pdc1,6,0,0,1,1,True,True
7,pdc1,7,0,0,1,1,True,True
8,pdc1,8,0,0,1,1,True,False
9,pdc1,9,1,0,0,1,True,True


In [ ]:
s = pd.Series([False, True, True, False, True, True, True, False, True])

grp = s.ne(s.shift()).cumsum()
tailles = s.groupby(grp).sum()

masque = grp == tailles.idxmax()
sequence = masque[masque]

debut = sequence.index[0]
fin   = sequence.index[-1]
longueur= fin-debut+1

debut, fin, longueur

(np.int64(4), np.int64(6), np.int64(3))

In [29]:
grp, masque

(0    1
 1    2
 2    2
 3    3
 4    4
 5    4
 6    4
 7    5
 8    6
 dtype: int64,
 0    False
 1    False
 2    False
 3    False
 4     True
 5     True
 6     True
 7    False
 8    False
 dtype: bool)

In [30]:
masque[masque]

4    True
5    True
6    True
dtype: bool

In [20]:
longueur_max = s.groupby(grp).sum().max()
longueur_max

np.int64(3)

In [21]:
tailles = s.groupby(grp).sum()
tailles

1    0
2    2
3    0
4    3
5    0
6    1
dtype: int64

In [22]:
idx = tailles.idxmax()
idx, tailles[idx]

(np.int64(4), np.int64(3))

In [23]:
masque = grp==idx
masque

0    False
1    False
2    False
3    False
4     True
5     True
6     True
7    False
8    False
dtype: bool

In [24]:
debut = masque.idxmax()
fin = debut + tailles[idx] - 1
debut, fin

(4, np.int64(6))